# 01 - Aperçu des sources

Aperçu des données produites par le simulateur et des fichiers DVF, avec les fonctions des pipelines (`pipelines/`). Rien n'est chargé en base.

In [1]:
import sys
from datetime import datetime, timezone

import pandas as pd

sys.path.append("../pipelines")

from dvf_pipeline import FILE_URL, read_dvf_csv  # noqa: E402
from fake_data import (  # noqa: E402
    generate_customers,
    generate_orders,
    generate_products,
)

## Back-office simulé

In [2]:
now = datetime.now(timezone.utc)

products = pd.DataFrame(generate_products())
customers = pd.DataFrame(generate_customers(now))
products.head()

,product_id,product_name,category,unit_price,unit_cost
0,1,Électronique Supporter,Électronique,214.41,165.58
1,2,Jardin Chute,Jardin,255.09,168.82
2,3,Sport Animal,Sport,78.41,45.55
3,4,Jardin Fin,Jardin,195.24,140.30
4,5,Maison Été,Maison,71.24,51.23


In [3]:
customers["segment"].value_counts()

segment
Standard    308
Premium     133
VIP          59
Name: count, dtype: int64

In [4]:
since = datetime(2026, 11, 1, tzinfo=timezone.utc)
orders = pd.DataFrame(generate_orders(updated_since=since, now=now))
orders["status"].value_counts()

status
delivered    604
shipped       87
pending       65
returned      48
cancelled     33
Name: count, dtype: int64

In [5]:
# Emails mal saisis, nettoyés dans stg_ecoshop__customers
dirty = customers["email"] != customers["email"].str.strip().str.lower()
customers.loc[dirty, ["customer_id", "email"]].head()

,customer_id,email
0,1,HELENESALMON@EXAMPLE.ORG
15,16,CHARLOTTE74@EXAMPLE.COM
50,51,VIRGINIEGAUDIN@EXAMPLE.NET
68,69,PEREIRASUZANNE@EXAMPLE.ORG
84,85,MARTINEBONNEAU@EXAMPLE.ORG


## Un fichier DVF (Paris, 2025)

In [6]:
import requests

url = FILE_URL.format(year=2025, department="75")
response = requests.get(url, timeout=120)
dvf = read_dvf_csv(response.content).to_pandas()
dvf.shape

(84440, 40)

In [7]:
dvf["type_local"].value_counts(dropna=False)

type_local
Dépendance                                  40655
Appartement                                 37796
Local industriel. commercial ou assimilé     5186
NaN                                           613
Maison                                        190
Name: count, dtype: int64

In [8]:
# Une mutation peut avoir plusieurs lignes avec le même prix
lines_per_mutation = dvf.groupby("id_mutation").size()
lines_per_mutation.describe()

count    40811.000000
mean         2.069050
std          5.365256
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max        481.000000
dtype: float64